In [1]:
import sys
import re
from pathlib import Path 
sys.path.append(str(Path().resolve().parents[0]))

import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from config import PROCESSED_DATA_DIR

In [2]:
reviews_df = pd.read_parquet(PROCESSED_DATA_DIR / "amazon_reviews_features.parquet")

In [3]:
def clean_text(text: str) -> str:
    """Prepare review text for a TF-IDF baseline."""
    text = str(text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


vectorizer = TfidfVectorizer(
    lowercase=False,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_features=10000
)

In [4]:
# Create cleaned text column for TF-IDF
reviews_df["tfidf_text"] = reviews_df["full_text"].apply(clean_text)

# Inspect a sample of the cleaned text
reviews_df[["full_text", "tfidf_text"]].head()

,full_text,tfidf_text
0,A Store That Doesn't Want to Sell Anything I r...,a store that doesn't want to sell anything i r...
1,Had multiple orders one turned up and… Had mul...,had multiple orders one turned up and had mult...
2,I informed these reprobates I informed these r...,i informed these reprobates i informed these r...
3,Advertise one price then increase it on websit...,advertise one price then increase it on websit...
4,If I could give a lower rate I would If I coul...,if i could give a lower rate i would if i coul...


In [5]:
# Split the data before fitting the TF-IDF vectorizer
train_df, test_df = train_test_split(
    reviews_df,
    test_size=0.2,
    random_state=42,
    stratify=reviews_df["rating"]
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.25,
    random_state=42,
    stratify=train_df["rating"]
)

# Fit on training text only, then transform validation and test text
X_train_tfidf = vectorizer.fit_transform(train_df["tfidf_text"])
X_val_tfidf = vectorizer.transform(val_df["tfidf_text"])
X_test_tfidf = vectorizer.transform(test_df["tfidf_text"])

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)
print("TF-IDF train matrix:", X_train_tfidf.shape)
print("TF-IDF validation matrix:", X_val_tfidf.shape)
print("TF-IDF test matrix:", X_test_tfidf.shape)

Train shape: (12633, 23)
Validation shape: (4211, 23)
Test shape: (4211, 23)
TF-IDF train matrix: (12633, 10000)
TF-IDF validation matrix: (4211, 10000)
TF-IDF test matrix: (4211, 10000)


In [6]:
# Save text-preprocessed splits for downstream modeling
train_df.to_parquet(PROCESSED_DATA_DIR / "amazon_reviews_train.parquet", index=False)
val_df.to_parquet(PROCESSED_DATA_DIR / "amazon_reviews_val.parquet", index=False)
test_df.to_parquet(PROCESSED_DATA_DIR / "amazon_reviews_test.parquet", index=False)